# Procesamiento de series temporales de MAVEN
### Cuaderno integrador — trabajo final de Procesamiento de Series Temporales (UBA)

**Problema físico:** caracterizar las mediciones de MAVEN y estudiar cómo se relacionan las variaciones del campo magnético y del flujo electrónico cerca del *bow shock* (BS) y la *magnetic pileup boundary* (MPB).

**Alcance:** reúne el trabajo realizado hasta ahora, desde control de calidad y tratamiento de gaps hasta cobertura espacial y mapas de correlación MAG–SWEA. Distingue los **resultados que ya están guardados** de los **análisis propuestos para etapas posteriores**.

**Principio de reproducibilidad:** este cuaderno **no consulta BigQuery ni vuelve a ejecutar automáticamente los procesos costosos**. Lee tablas CSV, figuras PNG y, opcionalmente, un único archivo MAG Parquet del repositorio local. Los cálculos originales están en los scripts citados en cada sección.

**Última actualización metodológica:** ventanas de correlación de **2 a 60 minutos**, incrementos de **1 minuto**; 64 energías de SWEA. Los resultados de correlación se cargan si están presentes localmente.


## Índice y cadena de procesamiento

1. Configuración y disponibilidad de archivos.
2. MAG: cobertura, calidad temporal, gaps internos y entre archivos.
3. Estudio de reconstrucción: variabilidad, pseudogaps, política de interpolación y validación.
4. SWEA: cubo de flujos, canales de energía y cadencia.
5. MAVEN: cobertura espacial y por mes.
6. Catálogo clasificado de BS/MPB y sincronización MAG–SWEA.
7. Correlaciones de Pearson dependientes de la energía y la duración de la ventana.
8. CWT Morlet de MAG centrada en MPB: cobertura, comparación pareada y archivo de resultados.
9. Limitaciones y siguientes análisis de series temporales.

**Convenciones:** tiempos UTC; componentes magnéticas MAG en nT; energía SWEA en eV; duración de la ventana **total** (no semiancho); mapas con promedio aritmético de coeficientes por cruce.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

# Funciona tanto si se abre Jupyter desde la raíz como desde notebooks/.
candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
ROOT = next((p for p in candidates if (p / "scripts" / "mag").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
print("Raíz del repositorio:", ROOT)

def cargar_csv(ruta, filas=8, mostrar=True, **kwargs):
    p = ROOT / ruta
    if not p.exists():
        print("No disponible:", ruta)
        return None
    df = pd.read_csv(p, **kwargs)
    print("{} | {} filas × {} columnas".format(ruta, len(df), len(df.columns)))
    if mostrar:
        display(df.head(filas))
    return df

def mostrar_figura(ruta, titulo=None):
    p = ROOT / ruta
    if titulo:
        display(Markdown("**" + titulo + "**"))
    if p.is_file():
        display(Image(filename=str(p)))
        return True
    print("Figura no disponible:", ruta)
    return False

def valor_metrica(df, nombre, columna_nombre="metric", columna_valor="value"):
    if df is None or columna_nombre not in df or columna_valor not in df:
        return None
    z = df.loc[df[columna_nombre].astype(str) == nombre, columna_valor]
    return z.iloc[0] if len(z) else None


### Disponibilidad del material

El notebook separa explícitamente **archivo ausente** de **resultado nulo**: si una figura o un CSV no está descargado, aparece un aviso, sin sustituirlo por datos inventados.

En particular, los resultados completos de **correlaciones** y sus Parquet de caché pueden existir en la computadora local aunque no estén publicados en GitHub. No se descargan aquí por su volumen.


In [ ]:
insumos = {
    "Resumen calidad MAG": "results/MAG_quality/mag_quality_summary.csv",
    "Gaps MAG internos": "results/MAG_gap_analysis/mag_gap_summary.csv",
    "Gaps entre archivos": "results/MAG_boundary_analysis/mag_boundary_summary.csv",
    "Pseudogaps": "results/MAG_pseudogap_validation/mag_pseudogap_winner_summary.csv",
    "Política interpolación": "results/MAG_gap_treatment_policy/mag_gap_treatment_summary.csv",
    "Tratamiento aplicado": "results/MAG_gap_treatment_applied/mag_gap_treatment_applied_summary.csv",
    "Control posterior MAG": "results/MAG_post_treatment_QC/mag_post_treatment_qc_summary.csv",
    "Energías SWEA": "results/SWEA_cadence/swea_energy_grid_summary.csv",
    "Cadencia SWEA": "results/SWEA_cadence/swea_cadence_by_energy.csv",
    "Cobertura espacial": "results/MAVEN_positions/position_summary.csv",
    "Correlación BS/MPB": "results/MAG_SWEA_boundary_correlation/MAVEN_MPB_4groups/correlations_by_boundary_summary.csv",
}
display(pd.DataFrame(
    [{"Análisis": k, "Disponible localmente": (ROOT / v).is_file(), "Archivo": v}
     for k, v in insumos.items()]
))


---

## 1. MAG: calidad y cobertura temporal

**Instrumento:** magnetómetro MAG. Variables principales: componentes \(B_x,B_y,B_z\) y módulo \(|B|\) en coordenadas SS.

**Script:** scripts/mag/01_MAG_quality_control.py.

**Resumen ya obtenido:** **3.839 días** (2014-10-11 a 2025-12-05), **329.705.412 muestras MAG**, mediana de cadencia diaria de **1 segundo**, sin muestras de campo inválidas ni timestamps duplicados en el conjunto auditado. Estos son resultados de la tabla de control de calidad almacenada, no una inferencia a partir de la figura.

Los huecos temporales se analizan por separado: una fracción válida del campo del 100 % no significa una serie sin interrupciones.


In [ ]:
qc = cargar_csv("results/MAG_quality/mag_quality_summary.csv", filas=1)
if qc is not None:
    campos = ["available_days", "first_date", "last_date", "total_samples",
              "weighted_valid_fraction", "median_of_daily_median_cadence_s",
              "total_duplicate_timestamps", "total_gaps"]
    display(qc[[x for x in campos if x in qc.columns]].T.rename(columns={0: "valor"}))
mostrar_figura("results/MAG_quality/figures/mag_dt_histogram_all_dates.png",
               "Distribución de intervalos entre muestras MAG")
mostrar_figura("results/MAG_quality/figures/mag_gaps_all_dates.png",
               "Gaps identificados en MAG")


## 2. MAG: gaps internos y discontinuidades entre archivos

**Scripts:**
- scripts/mag/01b_MAG_gap_analysis.py: gaps **dentro** de cada archivo fuente.
- scripts/mag/01c_MAG_boundary_analysis.py: continuidad y separación temporal **entre** archivos.

La distinción es metodológica: un salto entre el final de un archivo y el comienzo del siguiente no debe confundirse con un gap interno de un archivo.

**Resultados archivados:**
- **6.171 gaps internos**, mediana de separación de **6,499 s** y máximo de **41.404,354 s**.
- **2.883 gaps entre archivos** (según el resumen de continuidad); también se registró una anomalía de frontera.
- La etapa posterior de política audita **9.055 gaps** en total según su propio catálogo.

Los recuentos pueden diferir cuando se consideran gaps, discontinuidades, anomalías y bordes de archivo. Siempre se informa la definición y el origen del número utilizado.


In [ ]:
gaps = cargar_csv("results/MAG_gap_analysis/mag_gap_summary.csv", filas=1)
bordes = cargar_csv("results/MAG_boundary_analysis/mag_boundary_summary.csv", filas=18)
if gaps is not None:
    mostrar = ["gap_count", "median_gap_s", "p90_gap_s", "p99_gap_s",
               "max_gap_s", "estimated_missing_samples_total"]
    display(gaps[[c for c in mostrar if c in gaps.columns]].T)
mostrar_figura("results/MAG_gap_analysis/figures/mag_gap_duration_histogram.png",
               "Distribución de duración de gaps internos")
mostrar_figura("results/MAG_gap_analysis/figures/mag_gap_survival_curve.png",
               "Función de supervivencia de duración de gaps")
mostrar_figura("results/MAG_boundary_analysis/figures/mag_boundary_dt_histogram.png",
               "Continuidad temporal en los límites de archivo")


---

## 3. Tratamiento de gaps de MAG

### 3.1 Diagnóstico de la estructura del campo cerca de los gaps

**Script:** scripts/mag/01d_MAG_gap_interpolation_analysis.py.

Se compararon saltos relativos de campo, rotación entre vectores magnéticos y variabilidad local a ambos lados de gaps. El diagnóstico no modifica los datos; su función es estudiar cuándo una interpolación podría ser defendible y cuándo podría suavizar una transición física real.


In [ ]:
cargar_csv("results/MAG_interpolation_analysis/mag_gap_interpolation_candidate_summary.csv")
mostrar_figura("results/MAG_interpolation_analysis/figures/gap_duration_vs_relative_jump.png",
               "Salto relativo versus duración del gap")
mostrar_figura("results/MAG_interpolation_analysis/figures/gap_duration_vs_angle.png",
               "Rotación magnética a través de gaps")


### 3.2 Validación mediante pseudogaps

**Script:** scripts/mag/01e_MAG_pseudogap_validation.py.

Se toman intervalos continuos con datos reales, se ocultan artificialmente muestras y se reconstruyen con tres técnicas: **interpolación lineal**, **splines cúbicas** y **autorregresión local bidireccional**. Como los tres métodos reconstruyen los mismos puntos ocultos, es posible comparar sus errores sobre las mismas pruebas.

El experimento considera diferentes duraciones y regímenes de variabilidad, utilizando métricas del error vectorial y de dirección. El método más conveniente no se selecciona únicamente por la apariencia visual de una reconstrucción.

Los archivos guardados contienen fracciones de victoria por duración y errores por método; no se debe generalizar un único ganador a todos los regímenes.


In [ ]:
ganadores = cargar_csv("results/MAG_pseudogap_validation/mag_pseudogap_winner_summary.csv", filas=15)
if ganadores is not None and {"target_gap_duration_s", "winner_method", "win_fraction"}.issubset(ganadores):
    display(ganadores.pivot_table(index="target_gap_duration_s",
                                 columns="winner_method",
                                 values="win_fraction", aggfunc="first"))
mostrar_figura("results/MAG_pseudogap_validation/figures/normalized_vector_rmse_by_gap_method.png",
               "Error normalizado del vector B por duración y método")
mostrar_figura("results/MAG_pseudogap_validation/figures/error_vs_variability.png",
               "Error de reconstrucción y variabilidad local")
mostrar_figura("results/MAG_pseudogap_validation/figures/winner_fraction_by_gap.png",
               "Fracción de victorias de cada método")


### 3.3 Política validada de interpolación

**Script:** scripts/mag/01f_MAG_gap_treatment_policy.py.

Se define una medida de variabilidad local de la serie magnética y se utilizan dos umbrales obtenidos en el análisis de pseudogaps:

- **Quiet:** \(V \leq 0{,}0471873\); se puede interpolar si \(\Delta t \leq 5\) s.
- **Intermediate:** \(0{,}0471873 < V \leq 0{,}174766\); se puede interpolar si \(\Delta t \leq 3\) s.
- **Variable:** \(V > 0{,}174766\); no se interpola.

**Resultado archivado:** de **9.055 gaps** clasificados, **3.379** fueron aprobados y **5.676** quedaron sin interpolar.

El rechazo de una interpolación no significa que el dato sea inválido: significa que no se reconstruyen muestras que no pueden justificarse con la política definida.


In [ ]:
politica = cargar_csv("results/MAG_gap_treatment_policy/mag_gap_treatment_summary.csv", filas=20)
cargar_csv("results/MAG_gap_treatment_policy/mag_gap_treatment_summary_by_regime.csv")
mostrar_figura("results/MAG_gap_treatment_policy/figures/gap_treatment_by_regime.png",
               "Decisiones de tratamiento según régimen")


### 3.4 Aplicación y control posterior

**Scripts:** scripts/mag/01g_MAG_apply_gap_treatment.py y scripts/mag/01h_MAG_post_treatment_QC.py.

- La interpolación aprobada es **lineal para \(B_x,B_y,B_z\)**; se recalcula \(|B|\) a partir del vector.
- **Se conservan todos los timestamps y las mediciones originales**. Solo se insertan muestras nuevas para los gaps autorizados.
- Los gaps rechazados se conservan como intervalos faltantes explícitos; no se rellenan con puntos artificiales.
- Cada muestra informa si es **original** o **interpolada**.

**Resultado del procesamiento:** **4.433 muestras interpoladas** incorporadas a **329.705.412 muestras originales**.

**Control posterior:** **PASS**, sin tiempos duplicados, intervalos temporales no positivos ni fallas de validación de las interpolaciones en el resumen archivado.


In [ ]:
aplicado = cargar_csv("results/MAG_gap_treatment_applied/mag_gap_treatment_applied_summary.csv", filas=20)
posterior = cargar_csv("results/MAG_post_treatment_QC/mag_post_treatment_qc_summary.csv", filas=30)
if posterior is not None:
    print("Estado global QC:", valor_metrica(posterior, "overall_qc_status"))
mostrar_figura("results/MAG_post_treatment_QC/figures/gap_count_before_after.png",
               "Cantidad de gaps antes y después del tratamiento")
mostrar_figura("results/MAG_post_treatment_QC/figures/interpolated_samples_by_day.png",
               "Muestras interpoladas por día")


### 3.5 Ejemplo opcional: un día MAG tratado

Esta celda **no lee toda la misión**: abre únicamente un Parquet diario y muestra 10 minutos de \(|B|\), destacando las muestras interpoladas. Si no está en la computadora local, se omite.


In [ ]:
ejemplo = ROOT / "data/processed/MAG_gap_treated/2014/11/mag_2014-11-12.parquet"
if ejemplo.exists():
    cols = ["datetime_utc", "b_total_nt", "is_interpolated"]
    mag_ejemplo = pd.read_parquet(ejemplo, columns=cols)
    mag_ejemplo["datetime_utc"] = pd.to_datetime(mag_ejemplo["datetime_utc"], utc=True)
    t0 = mag_ejemplo["datetime_utc"].min()
    mag10 = mag_ejemplo.loc[mag_ejemplo["datetime_utc"] < t0 + pd.Timedelta(minutes=10)]
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(mag10["datetime_utc"], mag10["b_total_nt"], lw=0.8, label="|B|")
    relleno = mag10.loc[mag10["is_interpolated"].fillna(False)]
    if len(relleno):
        ax.scatter(relleno["datetime_utc"], relleno["b_total_nt"], s=18,
                   label="Interpolado", zorder=3)
    ax.set(xlabel="Fecha UTC", ylabel="|B| (nT)", title="Ejemplo MAG tratado: 10 min")
    ax.legend()
    fig.autofmt_xdate()
    plt.show()
else:
    print("No hay Parquet MAG local para este ejemplo:", ejemplo)


---

## 4. SWEA: resolución temporal y grilla energética

**Script:** scripts/swea/02a_SWEA_cadence_by_energy.py.

SWEA L2 svy3d mide una distribución de flujo de electrones en un cubo angular-energético. En los archivos utilizados se registran **6 elevaciones × 16 acimuts × 64 energías**. La correspondencia entre eje energético y orden del arreglo aplanado se verifica contra la tabla de ejes.

Se estudió la disponibilidad y la cadencia de cada energía usando solamente timestamps donde existe al menos un flujo angular válido para ese canal. El script estima la cadencia nominal a partir de los datos; **no la presupone en cada muestra**.

La tabla completa de cadencia puede estar disponible solo localmente si no se subió al repositorio.


In [ ]:
rejilla = cargar_csv("results/SWEA_cadence/swea_energy_grid_summary.csv",
                     filas=4, mostrar=False)
if rejilla is not None:
    print("Archivos SWEA examinados:", len(rejilla))
    print("Dimensiones de muestra:", rejilla[["n_elev","n_azim","n_energy"]].head(1).to_dict("records"))
    if "energy_grid_matches_reference" in rejilla:
        ok = rejilla["energy_grid_matches_reference"].astype(str).str.lower().eq("true")
        print("Grillas coincidentes:", int(ok.sum()), "/", len(rejilla))
cadencia = cargar_csv("results/SWEA_cadence/swea_cadence_by_energy.csv", filas=8)
cargar_csv("results/SWEA_cadence/swea_cadence_global_summary.csv")
mostrar_figura("results/SWEA_cadence/figures/swea_cadence_by_energy.png",
               "Cadencia temporal de SWEA por canal energético")


---

## 5. Cobertura espacial de las mediciones

**Scripts:**
- scripts/positions/03a_MAVEN_position_spatial_coverage.py
- scripts/positions/03b_MAVEN_frontal_by_month.py

Se utilizan posiciones de MAVEN en coordenadas SS almacenadas en BigQuery. Para visualizar la cobertura espacial se presenta la coordenada radial transversal con signo:

\[
\rho_{\mathrm{sign}}=\operatorname{sign}(z)\sqrt{y^2+z^2}.
\]

La proyección frontal Y–Z de cada año/mes representa **todas** las posiciones seleccionadas proyectadas sobre ese plano; **no** equivale a restringir los datos a \(X=0\).

Esta parte contextualiza qué regiones espaciales son observadas y cuáles están menos representadas.


In [ ]:
pos = cargar_csv("results/MAVEN_positions/position_summary.csv", filas=1)
if pos is not None:
    cols = ["first_date", "last_date", "total_rows", "pct_valid_positions",
            "altitude_min_km", "altitude_max_km", "coordinate_frame"]
    display(pos[[c for c in cols if c in pos.columns]].T)
mostrar_figura("results/MAVEN_positions/figures/position_x_signed_rho_coverage.png",
               "Cobertura en el plano X–rho con signo")
mostrar_figura("results/MAVEN_positions/figures/position_y_z_coverage.png",
               "Cobertura frontal Y–Z")


In [ ]:
# Elegir un año para ver la cobertura frontal mensual; no vuelve a consultar BigQuery.
anio_ejemplo = 2016
mostrar_figura(
    "results/MAVEN_positions_monthly_front/figures/frontal_y_z_by_month_{}.png".format(anio_ejemplo),
    "Cobertura frontal mensual: {}".format(anio_ejemplo)
)
resumen_anual = cargar_csv("results/MAVEN_positions_monthly_front/frontal_y_z_year_summary.csv",
                          filas=15)


---

## 6. Catálogo de fronteras BS/MPB

**Fuente de los tiempos de cruce:** archivos etiquetados manualmente MAVEN_MPB_Data_Group*.csv, en la carpeta local **data/classified_crossings/**.

En el conjunto empleado para este estudio se trabajó con **569 casos** (cada uno con un BS y una MPB), es decir **1.138 anotaciones de frontera** antes de considerar posibles duplicados en el mismo tiempo y tipo. Los tiempos originales se preservan; las inconsistencias se registran en el archivo de auditoría.

No se sustituyen por cruces estimados a partir de la geometría del modelo de Vignes.

Para cada evento se extrae una ventana **centrada** en el instante clasificado; su duración total se varía para medir la dependencia de la correlación con la escala temporal.


In [ ]:
carpeta_catalogo = ROOT / "data/classified_crossings"
archivos_catalogo = sorted(carpeta_catalogo.glob("MAVEN_MPB_Data_Group*.csv"))
print("Catálogos locales encontrados:", len(archivos_catalogo))
for p in archivos_catalogo:
    print(" -", p.name)
base = "results/MAG_SWEA_boundary_correlation/MAVEN_MPB_4groups/"
eventos = cargar_csv(base + "events_selected.csv", filas=6)
alertas = cargar_csv(base + "catalog_issues.csv", filas=12)
if eventos is not None and "event_type" in eventos:
    display(eventos["event_type"].value_counts().rename("anotaciones").to_frame())


## 7. Sincronización MAG–SWEA

**Script:** scripts/correlations/04a_boundary_energy_correlation_maps.py.

1. SWEA se extrae de las tablas BigQuery **maven.swea_svy3d** y **maven.swea_svy3d_axes**, alrededor de los eventos clasificados.
2. En BigQuery se calcula para cada energía el **flujo medio sobre los canales angulares válidos**.
3. MAG proviene de los Parquet diarios **ya tratados**.
4. Para comparar las señales, \(B_x,B_y,B_z\) se interpolan **a los timestamps originales de SWEA**, sin remuestrear de nuevo SWEA.
5. Solo se aceptan coincidencias exactas o interpolaciones entre muestras MAG válidas con separación **\(\leq 1{,}5\ \mathrm{s}\)**. No hay extrapolación ni interpolación a través de gaps rechazados.
6. Se reconstruye \(|B|\) a partir del vector interpolado y se conserva una bandera que identifica cuando se usó MAG reconstruido.

Esta sincronización es parte del método de correlación, no un supuesto de que ambos instrumentos tengan la misma cadencia.


### 7.1 Correlación de Pearson según energía y duración

Para cada cruce, canal energético \(E\) y duración total \(\Delta t\):

\[
r(E,\Delta t)=\operatorname{corr}\left(
|B(t)|,\ \log_{10} J(E,t)
\right),
\quad t\in\left[t_c-\frac{\Delta t}{2},\ t_c+\frac{\Delta t}{2}\right].
\]

**Configuración por defecto actual:** \(\Delta t=2,3,\dots,60\) minutos (**59 ventanas**), **64 energías**, al menos **6 pares temporalmente válidos** para calcular un coeficiente. Si la varianza es cero o no hay suficientes pares, se registra **NaN**.

Se calcula primero \(r\) para cada evento y luego una **media aritmética de \(r\) entre eventos**, por separado para BS y MPB. Los coeficientes de distintos cruces no se obtienen concatenando todos los puntos. Se presentan también mapas con el número de eventos válidos por celda.

**Importante:** una media de correlaciones individuales es una estadística descriptiva; no equivale a un contraste de significancia ni a una correlación combinada ponderada.


In [ ]:
config_path = ROOT / base / "analysis_config.json"
if config_path.exists():
    conf = json.loads(config_path.read_text(encoding="utf-8"))
    print("Parámetros de la última ejecución local:")
    display(pd.Series({k: v for k, v in conf.items() if k != "events"}, name="valor").to_frame())
else:
    print("No hay configuración local de la correlación; las ventanas predeterminadas del script son 2 a 60 min.")
resumen_r = cargar_csv(base + "correlations_by_boundary_summary.csv", filas=8)
resumen_grupos = cargar_csv(base + "correlations_by_group_summary.csv", filas=8, mostrar=False)
if resumen_r is not None:
    for tipo in sorted(resumen_r["event_type"].dropna().unique()):
        sub = resumen_r.loc[resumen_r["event_type"] == tipo]
        print("{}: {} ventanas; {} energías; {} celdas con r definido".format(
            tipo, sub["window_minutes"].nunique(), sub["energy_index"].nunique(),
            sub["mean_pearson_r"].notna().sum()))


### 7.2 Mapas promedio: BS y MPB

**Eje X:** energías SWEA reales (eV, escala logarítmica). **Eje Y:** duración total de la ventana (minutos). **Color:** promedio de Pearson entre los cruces con coeficiente definido.

Los gráficos originales se cargan del directorio de resultados de la computadora. Si no están todavía generados, esta celda lo informa sin solicitar datos a BigQuery.


In [ ]:
for frontera in ("BS", "MPB"):
    mostrar_figura(base + "figures/summary/{}_b_total_nt_mean_r.png".format(frontera),
                   "Correlación promedio: {}".format(frontera))


### 7.3 ¿Cuántos cruces contribuyen a cada celda?

La robustez de un mapa no se puede evaluar mirando solo el color del coeficiente. El número de eventos válidos \(N(E,\Delta t)\) debe considerarse para evitar interpretar como patrón general una celda basada en pocos cruces.


In [ ]:
for frontera in ("BS", "MPB"):
    mostrar_figura(base + "figures/summary/{}_b_total_nt_n_cases.png".format(frontera),
                   "Cantidad de cruces válidos: {}".format(frontera))


### 7.4 Reproducir mapas a partir de las tablas compactas (sin BigQuery)

Esta celda ofrece una **visualización alternativa** si están disponibles las tablas de resumen. No lee el CSV individual completo (que puede tener millones de filas).

Se utilizan exactamente los valores **mean_pearson_r** y **n_valid_events** ya calculados y guardados.


In [ ]:
def mapa_desde_resumen(df, frontera, columna="mean_pearson_r", campo="b_total_nt"):
    if df is None:
        print("Falta el CSV de resumen; no se recalcula la correlación.")
        return
    f = df.loc[(df["event_type"] == frontera) & (df["field"] == campo)].copy()
    if f.empty:
        print("Sin datos de", frontera, campo)
        return
    tabla = f.pivot_table(index="window_minutes", columns="energy_index",
                          values=columna, aggfunc="first").sort_index().sort_index(axis=1)
    energias = (f[["energy_index", "energy_ev"]].drop_duplicates()
                  .set_index("energy_index").loc[tabla.columns, "energy_ev"].to_numpy(float))
    x = np.asarray(energias, dtype=float)
    # SWEA puede almacenar energías en orden descendente; graficar de menor a mayor.
    orden = np.argsort(x)
    x = x[orden]
    tabla = tabla.iloc[:, orden]
    y = tabla.index.to_numpy(dtype=float)
    if not (len(y) and len(x)):
        return
    # Intervalos centrados en energía, usando bordes geométricos para escala log.
    x_edges = np.empty(len(x) + 1)
    x_edges[1:-1] = np.sqrt(x[:-1] * x[1:])
    x_edges[0] = x[0] / np.sqrt(x[1] / x[0])
    x_edges[-1] = x[-1] * np.sqrt(x[-1] / x[-2])
    y_edges = np.empty(len(y) + 1)
    y_edges[1:-1] = (y[:-1] + y[1:]) / 2
    y_edges[0] = y[0] - (y[1] - y[0]) / 2
    y_edges[-1] = y[-1] + (y[-1] - y[-2]) / 2
    fig, ax = plt.subplots(figsize=(13, 5))
    kwargs = {"vmin": -1, "vmax": 1, "cmap": "coolwarm"} if columna == "mean_pearson_r" else {"cmap": "viridis"}
    mesh = ax.pcolormesh(x_edges, y_edges, tabla.to_numpy(float), shading="flat", **kwargs)
    ax.set_xscale("log")
    ax.set(xlabel="Energía SWEA [eV]", ylabel="Duración total [min]",
           title="{} — {}".format(frontera, columna))
    fig.colorbar(mesh, ax=ax)
    plt.show()

for frontera in ("BS", "MPB"):
    mapa_desde_resumen(resumen_r, frontera)


### 7.5 Dependencia de la correlación con la ventana temporal

Para evaluar visualmente una eventual **meseta**, se puede extraer \(r\) como función de \(\Delta t\) a energías seleccionadas, usando la media ya guardada por frontera.

No se fuerza la existencia de una meseta: la conclusión debe surgir de los datos y contemplar la variabilidad entre eventos.


In [ ]:
if resumen_r is None:
    print("Se necesita correlations_by_boundary_summary.csv para esta comparación.")
else:
    objetivo_eV = [50, 100, 200]
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
    for ax, frontera in zip(axs, ("BS", "MPB")):
        f = resumen_r.loc[(resumen_r["event_type"] == frontera) &
                          (resumen_r["field"] == "b_total_nt")].copy()
        canales = f[["energy_index", "energy_ev"]].drop_duplicates()
        for objetivo in objetivo_eV:
            if canales.empty:
                continue
            idx = (canales["energy_ev"].astype(float) - objetivo).abs().idxmin()
            canal = int(canales.loc[idx, "energy_index"])
            energia_real = float(canales.loc[idx, "energy_ev"])
            muestra = f.loc[f["energy_index"] == canal].sort_values("window_minutes")
            ax.plot(muestra["window_minutes"], muestra["mean_pearson_r"],
                    marker=".", label="{:.1f} eV".format(energia_real))
        ax.axhline(0, lw=0.8, color="gray")
        ax.set(title=frontera, xlabel="Duración total [min]")
        ax.legend()
    axs[0].set_ylabel("Media de Pearson r")
    fig.suptitle("Evolución de r según el tamaño de la ventana")
    plt.tight_layout()
    plt.show()


---

## 8. Transformada wavelet continua (CWT) de MAG en torno a la MPB

**Objetivo:** examinar la estructura tiempo–escala de las fluctuaciones magnéticas en cada cruce MPB clasificado, con una **ventana total de 10 minutos**, centrada en el instante MPB (\(-5\) a \(+5\) min).

**Implementación:** scripts/wavelets/05a_MPB_MAG_wavelet_10min.py.

**Decisiones metodológicas:**

- Serie: \(|B|\) de **MAG tratado**, en nT. Se pueden estudiar también \(B_x\), \(B_y\) y \(B_z\) cambiando \`--field\`.
- Wavelet compleja Morlet: \`cmor1.5-1.0\` (PyWavelets). Resolución nominal: 1 s, con períodos analizados entre **4 y 120 s** (48 escalas logarítmicas).
- Se quita una tendencia lineal de cada tramo continuo antes de aplicar la CWT. Para evitar sesgos debidos a amplitudes diferentes, la potencia \(|W|^2\) se normaliza por la varianza del residuo del evento.
- **Sin cubrir gaps largos:** los tramos de MAG se separan cuando la diferencia entre timestamps consecutivos válidos supera 1,5 s. Cada segmento se interpola a una grilla de 1 s *por separado*. No se interpolan muestras en discontinuidades rechazadas.
- **Cono de influencia (COI):** se excluyen las zonas cercanas a los bordes de cada segmento, donde la CWT depende de la extensión de la señal. El enmascaramiento es una aproximación conservadora de la envolvente Morlet empleada.
- Cada cruce tiene su propio escalograma. Para el mapa global se calcula la **media de potencia normalizada** entre cruces con datos válidos en cada punto tiempo–período y se guarda también el número de cruces que contribuyeron.

Una región de alta potencia cerca de \(t=0\) **no demuestra una onda**: la propia discontinuidad de la MPB puede producir potencia de banda ancha. No se ha calculado significancia frente a ruido rojo; este estudio es exploratorio.


### 8.1 Resultados de la CWT sobre los cruces MPB

El proceso por lotes escribe un CSV con indicadores por cruce, otro con potencia por período, un mapa global promedio y una selección de escalogramas individuales.

Si falta alguna salida, el notebook calcula las wavelets de todos los cruces MPB usando los Parquet locales (sin BigQuery); si existen todas, las reutiliza.


### 8.1.a Generar los resultados wavelet desde Jupyter

El notebook comprueba automáticamente si están disponibles los resultados del análisis de 10 minutos alrededor de las MPB, incluida la nueva **tabla pareada antes/después por período y evento**. Si falta alguno, ejecuta el script para todos los cruces, **sin acceder a BigQuery**, y luego reutiliza las salidas. La nueva tabla exige una segunda ejecución local del script una sola vez aunque los gráficos anteriores ya existan. Para desactivar el cálculo elegí \`EJECUTAR_WAVELETS="nunca"\`.


In [ ]:
import sys
import subprocess

wavelet_path = ROOT / "results/MAG_MPB_wavelets_10min"
wavelet_dir = "results/MAG_MPB_wavelets_10min/"
catalogo_wavelet_csv = ROOT / "data/classified_crossings/crossings_catalog_normalized_4groups.csv"
catalogo_wavelet = (catalogo_wavelet_csv if catalogo_wavelet_csv.is_file()
                    else ROOT / "data/classified_crossings")
script_wavelet = ROOT / "scripts/wavelets/05a_MPB_MAG_wavelet_10min.py"
EJECUTAR_WAVELETS = "si_falta"  # opciones: "si_falta", "siempre", "nunca"

necesarios = [
    "mpb_wavelet_event_summary.csv",
    "mpb_wavelet_period_by_event.csv",
    "mpb_wavelet_before_after_by_event.csv",
    "mpb_wavelet_mean_map.csv",
    "analysis_config.json",
    "figures/mpb_mean_wavelet_power.png",
]
faltantes = [s for s in necesarios if not (wavelet_path / s).is_file()]
print("Salidas wavelet faltantes:", faltantes if faltantes else "ninguna")
ejecutar = (EJECUTAR_WAVELETS == "siempre" or
            (EJECUTAR_WAVELETS == "si_falta" and len(faltantes) > 0))
if ejecutar:
    if not script_wavelet.is_file() or not catalogo_wavelet.exists():
        raise FileNotFoundError("Falta el script wavelet o el catálogo de cruces MPB.")
    print("Calculando CWT de los cruces MPB con datos MAG locales...", flush=True)
    cmd = [sys.executable, str(script_wavelet),
           "--events", str(catalogo_wavelet),
           "--mag-dir", str(ROOT / "data/processed/MAG_gap_treated"),
           "--output-root", str(wavelet_path),
           "--max-plots", "12"]
    ejecucion = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True)
    print("\n".join(ejecucion.stdout.splitlines()[-20:]))
    if ejecucion.returncode:
        print(ejecucion.stderr[-3500:])
        raise RuntimeError("Falló el cálculo wavelet, código {}".format(ejecucion.returncode))
    print("Wavelets calculadas y resultados guardados en:", wavelet_path)
else:
    print("Reutilizando resultados locales; no se consultó BigQuery.")


In [ ]:
wavelet_dir = "results/MAG_MPB_wavelets_10min/"
cwt_resumen = cargar_csv(wavelet_dir + "mpb_wavelet_event_summary.csv", filas=10)
cwt_periodos = cargar_csv(wavelet_dir + "mpb_wavelet_period_by_event.csv",
                          filas=8, mostrar=False)
cwt_map = cargar_csv(wavelet_dir + "mpb_wavelet_mean_map.csv",
                     filas=5, mostrar=False)
cwt_pareado = cargar_csv(wavelet_dir + "mpb_wavelet_before_after_by_event.csv",
                        filas=8, mostrar=False)
if cwt_resumen is not None and "status" in cwt_resumen.columns:
    print("Estado de los eventos:")
    display(cwt_resumen["status"].value_counts(dropna=False).to_frame("eventos"))
    if "dominant_period_s" in cwt_resumen:
        display(cwt_resumen["dominant_period_s"].describe().to_frame("período dominante [s]"))
mostrar_figura(wavelet_dir + "figures/mpb_mean_wavelet_power.png",
               "Potencia wavelet normalizada promedio alrededor de la MPB")


### 8.2 Ejemplo reproducible: CWT de un cruce MPB individual

Esta celda **sí calcula** la CWT para **un solo cruce** con datos MAG locales (sin BigQuery); prueba hasta 20 eventos del catálogo ordenado hasta encontrar uno con datos y potencia válida fuera del COI.

No ejecuta el análisis completo de cientos de eventos. Guarda la figura en \`results/MAG_MPB_wavelets_10min/notebook_example.png\`.


In [ ]:
import importlib.util
ruta_wavelet = ROOT / "scripts/wavelets/05a_MPB_MAG_wavelet_10min.py"
catalogo_wavelet = ROOT / "data/classified_crossings"
if not catalogo_wavelet.is_dir():
    catalogo_wavelet = ROOT / base / "events_selected.csv"

if not ruta_wavelet.exists():
    print("No se encuentra el script wavelet; ejecutá git pull origin main.")
elif not catalogo_wavelet.exists():
    print("Falta el catálogo original y events_selected.csv.")
else:
    spec = importlib.util.spec_from_file_location("mpb_cwt", str(ruta_wavelet))
    mpb_cwt = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mpb_cwt)
    eventos_cwt = mpb_cwt.load_mpb_catalog(catalogo_wavelet)
    ruta_mag = ROOT / "data/processed/MAG_gap_treated"
    tiempos_s = np.arange(-300., 301., 1.)
    periodos_s = np.geomspace(4., 120., 48)
    figura_cwt = ROOT / wavelet_dir / "notebook_example.png"
    ejemplo_cwt = None
    for ev in eventos_cwt.head(20).itertuples(index=False):
        try:
            def leer_dia(dia):
                return mpb_cwt.read_day(pd.Timestamp(dia, tz="UTC"),
                                         ruta_mag, "b_total_nt")
            datos = mpb_cwt.extract_mag_window(ev.mpb_utc, ruta_mag,
                                                  "b_total_nt", 300., leer_dia)
            senial, potencia, pot_por_periodo, n_por_periodo, estadisticas = (
                mpb_cwt.cwt_event(datos, ev.mpb_utc, "b_total_nt",
                                  tiempos_s, periodos_s))
            if estadisticas["status"] == "ok":
                mpb_cwt.plot_map(tiempos_s/60., periodos_s, potencia,
                                 "MPB | b_total_nt | {}".format(ev.mpb_utc),
                                 figura_cwt, signal=senial)
                ejemplo_cwt = ev
                print("Cruce seleccionado:", ev.case_id, ev.mpb_utc)
                print("Indicadores:", estadisticas)
                display(Image(filename=str(figura_cwt)))
                break
        except (FileNotFoundError, OSError, KeyError, ValueError) as err:
            print("Sin MAG disponible para", ev.mpb_utc, ":", err)
    if ejemplo_cwt is None:
        print("No se encontró un ejemplo con potencia válida entre los 20 primeros cruces.")


### 8.3 Ejecución equivalente desde la terminal (opcional)

Desde la **raíz del repositorio**, primero se puede hacer una prueba que no reemplace el resumen principal:

~~~bash
python3 scripts/wavelets/05a_MPB_MAG_wavelet_10min.py \
    --events data/classified_crossings/ \
    --max-events 5 --run-name prueba_5_mpb --max-plots 5
~~~

Luego, para analizar **todos** los cruces MPB clasificados (guardando 12 figuras individuales representativas):

~~~bash
python3 scripts/wavelets/05a_MPB_MAG_wavelet_10min.py \
    --events data/classified_crossings/ \
    --max-plots 12
~~~

Las tablas resumen contienen \`n_valid_events\`, que debe inspeccionarse junto con el mapa promedio. La marca \`bs_within_window\` permite detectar ventanas MPB que también contienen el BS anotado, sin descartarlas silenciosamente. Usar \`--exclude-interpolated\` para repetir el análisis omitiendo las muestras MAG reconstruidas.


### 8.4 Comprobar la cobertura del mapa

La CWT se enmascara cerca de los extremos de cada segmento mediante el COI. El mapa de **cantidad de MPB que contribuyen por píxel** permite distinguir ausencia de datos de una potencia baja. Se guarda el PNG para compartir.


In [ ]:
informe_wavelet = wavelet_path / "informe_notebook"
informe_wavelet.mkdir(parents=True, exist_ok=True)

if cwt_map is None:
    print("No está disponible el CSV del mapa wavelet.")
else:
    tabla = cwt_map.pivot(index="period_s", columns="time_relative_min",
                          values="n_valid_events").sort_index().sort_index(axis=1)
    t = tabla.columns.to_numpy(dtype=float)
    p = tabla.index.to_numpy(dtype=float)
    xt = np.r_[t[0]-(t[1]-t[0])/2, (t[1:]+t[:-1])/2, t[-1]+(t[-1]-t[-2])/2]
    yp = np.r_[p[0]/np.sqrt(p[1]/p[0]), np.sqrt(p[1:]*p[:-1]),
                p[-1]*np.sqrt(p[-1]/p[-2])]
    fig, ax = plt.subplots(figsize=(12,5))
    mesh = ax.pcolormesh(xt, yp, tabla.to_numpy(float), shading="flat",
                          vmin=0, vmax=cwt_map["n_valid_events"].max())
    ax.set_yscale("log")
    ax.axvline(0, ls="--", color="gray")
    ax.set(xlabel="Tiempo relativo a MPB [min]", ylabel="Período [s]",
           title="Número de eventos MPB válidos por píxel (fuera del COI)")
    fig.colorbar(mesh, ax=ax, label="N eventos válidos")
    fig.tight_layout()
    fig.savefig(informe_wavelet/"01_cobertura.png", dpi=180)
    plt.show()
    display(cwt_map["n_valid_events"].describe().to_frame())


### 8.5 Comparación pareada: antes y después de la MPB

En cada cruce y para cada período \(T\), se promedia la potencia wavelet **antes** (\(-5 \le t <0\) min) y **después** (\(0<t\le5\) min), excluyendo \(t=0\). Cada lado requiere por lo menos **20 tiempos válidos fuera del COI**.

Se calcula la diferencia logarítmica \(d_i(T)=\log_{10}[P_{i,\mathrm{después}}(T)/P_{i,\mathrm{antes}}(T)]\). Para cada período se presenta la **mediana de los mismos eventos** y un intervalo de incertidumbre bootstrap percentil al 95 % (remuestreando eventos, descriptivo). No se cuentan los 48 períodos como observaciones independientes.

**Interpretación:** el resultado distingue *antes/después temporalmente*, **no** magnetofunda frente a magnetosfera inducida: para compararlo por región física será necesario identificar cruces inbound y outbound.


In [ ]:
def ic_bootstrap_mediana(x, semilla=20261125, n_boot=600):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    central = float(np.median(x))
    if len(x) < 3:
        return central, np.nan, np.nan
    rng = np.random.default_rng(semilla)
    indices = rng.integers(len(x), size=(n_boot, len(x)))
    lim = np.quantile(np.median(x[indices], axis=1), [0.025, 0.975])
    return central, float(lim[0]), float(lim[1])

resumen_periodos = None
if cwt_pareado is None:
    print("Falta la tabla pareada. Revisá la celda 8.1.a.")
else:
    required = ["event_id", "period_s", "power_before", "power_after",
                "n_valid_before", "n_valid_after", "log10_after_over_before"]
    if not set(required).issubset(cwt_pareado.columns):
        raise ValueError("La tabla pareada no contiene las columnas esperadas.")
    pares_validos = cwt_pareado.loc[
        cwt_pareado["log10_after_over_before"].notna()].copy()
    print("MPB con pares válidos:", pares_validos.event_id.nunique())
    print("Pares (evento, período):", len(pares_validos))
    filas = []
    for j, (periodo, d) in enumerate(pares_validos.groupby("period_s")):
        x = d["log10_after_over_before"].to_numpy(float)
        med, li, ls = ic_bootstrap_mediana(x, semilla=20261125+j)
        filas.append(dict(period_s=periodo, n_paired_events=len(x),
                          median_log10_ratio=med, ci95_low=li, ci95_high=ls,
                          median_power_ratio=10**med, fraction_after_greater=float(np.mean(x>0))))
    resumen_periodos = pd.DataFrame(filas).sort_values("period_s")
    resumen_periodos.to_csv(informe_wavelet/"02_resumen_pareado_periodos.csv", index=False)
    display(resumen_periodos.head(12))
    xx = resumen_periodos["period_s"].to_numpy(float)
    fig, (ax, ax2) = plt.subplots(2,1,figsize=(11,7),sharex=True,
                                  gridspec_kw={"height_ratios":[2,1]})
    ax.plot(xx, resumen_periodos["median_log10_ratio"], label="Mediana de cruces")
    ax.fill_between(xx, resumen_periodos["ci95_low"],
                    resumen_periodos["ci95_high"], alpha=0.25, label="IC bootstrap 95 %")
    ax.axhline(0,ls="--",color="gray")
    ax.set(xscale="log", ylabel="Mediana log10(P después / P antes)",
           title="Potencia wavelet pareada antes/después de MPB")
    ax.legend()
    ax2.plot(xx, resumen_periodos["n_paired_events"])
    ax2.set(xscale="log", ylabel="N eventos pareados", xlabel="Período [s]")
    fig.tight_layout()
    fig.savefig(informe_wavelet/"02_potencia_pareada_periodos.png",dpi=180)
    plt.show()


### 8.6 Comparación por bandas de períodos

Se agrupan los períodos en **4–12, 12–30, 30–60 y 60–120 s**. Primero se calcula la mediana de las diferencias logarítmicas **dentro de cada evento y banda**, exigiendo al menos tres períodos válidos. Luego se compara entre eventos con intervalos bootstrap descriptivos. Se informa por separado la cantidad de eventos y los cuatro grupos clasificados.


In [ ]:
bandas = [(4.,12.), (12.,30.), (30.,60.), (60.,120.0001)]
tabla_eventos_banda = None
resumen_bandas = None
if cwt_pareado is None:
    print("No hay datos pareados para comparar bandas.")
else:
    filas=[]
    for inf, sup in bandas:
        nombre = "{}–{} s".format(int(inf), min(120, int(sup)))
        d = cwt_pareado.loc[(cwt_pareado.period_s>=inf) &
                            (cwt_pareado.period_s<sup) &
                            cwt_pareado.log10_after_over_before.notna()]
        for (eid, grp, bs), e in d.groupby(
                ["event_id","group","bs_within_window"],dropna=False):
            if len(e)>=3:
                filas.append(dict(event_id=eid, group=grp,
                                  bs_within_window=bs, banda_s=nombre,
                                  n_periods=len(e),
                                  median_log10_ratio=float(e.log10_after_over_before.median())))
    tabla_eventos_banda = pd.DataFrame(filas)
    tabla_eventos_banda.to_csv(informe_wavelet/"03_band_por_evento.csv", index=False)
    salidas=[]
    for j,(nombre,g) in enumerate(tabla_eventos_banda.groupby("banda_s")):
        x = g.median_log10_ratio.to_numpy(float)
        med,li,ls = ic_bootstrap_mediana(x,semilla=2026+j)
        salidas.append(dict(banda_s=nombre, n_events=len(x),
                            median_log10_ratio=med, median_power_ratio=10**med,
                            ci95_low=li, ci95_high=ls,
                            fraction_after_greater=float(np.mean(x>0))))
    resumen_bandas = pd.DataFrame(salidas)
    resumen_bandas.to_csv(informe_wavelet/"03_resumen_bandas.csv",index=False)
    display(resumen_bandas)
    print("Resumen descriptivo por grupo:")
    display(tabla_eventos_banda.groupby(["banda_s","group"])
            ["median_log10_ratio"].agg(["count","median"]))
    fig,ax=plt.subplots(figsize=(10,5))
    orden=["4–12 s","12–30 s","30–60 s","60–120 s"]
    q=resumen_bandas.set_index("banda_s").reindex(orden)
    x=np.arange(len(q))
    y=q.median_log10_ratio.to_numpy(float)
    ax.errorbar(x,y,yerr=[y-q.ci95_low.to_numpy(float),
                           q.ci95_high.to_numpy(float)-y],fmt="o",capsize=4)
    ax.axhline(0,ls="--",color="gray")
    ax.set(xticks=x,xticklabels=orden, ylabel="Mediana log10(P después / P antes)",
           xlabel="Banda temporal", title="Cambio de potencia por banda de períodos")
    fig.tight_layout()
    fig.savefig(informe_wavelet/"03_potencia_por_banda.png",dpi=180)
    plt.show()


### 8.7 Sensibilidad a la presencia del BS en la ventana MPB

Se compara el resultado usando todos los cruces y excluyendo aquellos cuya ventana de \(\pm5\) minutos contiene el BS clasificado. La ausencia del BS dentro de esta ventana **no garantiza que corresponda a una región homogénea**.


In [ ]:
if tabla_eventos_banda is None or tabla_eventos_banda.empty:
    print("No se puede evaluar la sensibilidad a BS.")
else:
    bs_flag = tabla_eventos_banda.bs_within_window.astype(str).str.lower().eq("true")
    print("Eventos con BS en ±5 min:", tabla_eventos_banda.loc[bs_flag,"event_id"].nunique())
    filas=[]
    for banda, bloque in tabla_eventos_banda.groupby("banda_s"):
        sin_bs = ~bloque.bs_within_window.astype(str).str.lower().eq("true")
        for etiqueta, d in [("todos", bloque),("sin_BS",bloque.loc[sin_bs])]:
            v=d.median_log10_ratio.to_numpy(float)
            filas.append(dict(banda_s=banda, seleccion=etiqueta,
                              n_events=len(v),
                              median_log10_ratio=float(np.median(v)) if len(v) else np.nan))
    sensibilidad_bs = pd.DataFrame(filas)
    display(sensibilidad_bs)
    sensibilidad_bs.to_csv(informe_wavelet/"04_sensibilidad_BS.csv",index=False)


### 8.8 Mostrar escalogramas individuales y exportar todas las salidas

Se muestran tres ejemplos individuales y se crea el archivo **\`MPB_wavelets_resultados_para_compartir.zip\`**. Incluye los CSV originales, gráficos de cobertura y comparaciones, resultados pareados, parámetros de ejecución y hasta 12 escalogramas individuales. El ZIP **no incluye los Parquet MAG** y puede subirse directamente a ChatGPT para analizar los resultados.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

eventos_png = sorted((wavelet_path/"figures/by_event").glob("*.png"))
print("Escalogramas individuales guardados:",len(eventos_png))
if eventos_png:
    indices={0,len(eventos_png)//2,len(eventos_png)-1}
    for j in sorted(indices):
        display(Markdown("**{}**".format(eventos_png[j].name)))
        display(Image(filename=str(eventos_png[j])))

archivos = [
    wavelet_path / nombre for nombre in [
        "analysis_config.json",
        "mpb_wavelet_event_summary.csv",
        "mpb_wavelet_period_by_event.csv",
        "mpb_wavelet_before_after_by_event.csv",
        "mpb_wavelet_mean_map.csv",
        "figures/mpb_mean_wavelet_power.png",
        "notebook_example.png",
    ]
]
archivos += sorted(informe_wavelet.glob("*.csv"))
archivos += sorted(informe_wavelet.glob("*.png"))
archivos += eventos_png[:12]
archivos = list(dict.fromkeys(f for f in archivos if f.is_file()))
zip_wavelet = wavelet_path/"MPB_wavelets_resultados_para_compartir.zip"
explicacion = (
    "MAVEN MAG: CWT Morlet (cmor1.5-1.0), ventana total 10 min en MPB.\n"
    "Períodos 4-120 s; gaps>1.5 s separados, COI enmascarado.\n"
    "Potencia normalizada por evento. Comparación pareada temporal:\n"
    "log10(P después / P antes), excluye t=0; mínimo 20 tiempos válidos por lado.\n"
    "Bootstrap de la mediana: descriptivo, NO test de significancia de ondas.\n"
    "Sin clasificación inbound/outbound, antes/después no significa región física.\n"
)
with ZipFile(zip_wavelet,"w",ZIP_DEFLATED,compresslevel=6) as z:
    z.writestr("LEER_PRIMERO.txt",explicacion)
    for f in archivos:
        z.write(f,arcname=str(f.relative_to(wavelet_path)))
print("\nARCHIVO PARA COMPARTIR:",zip_wavelet)
print("Tamaño: {:.1f} MiB".format(zip_wavelet.stat().st_size/1024**2))
print("Archivos incluidos:",len(archivos)+1)


---

## 9. Interpretación física y límites de la comparación

1. **No estacionariedad:** una correlación elevada en una ventana centrada en una frontera puede reflejar el cambio de nivel entre regiones, sin implicar que las fluctuaciones rápidas estén relacionadas.
2. **Separación BS–MPB:** una ventana centrada de hasta **60 minutos** puede incluir ambas fronteras. El script actual **no excluye automáticamente** esos casos.
3. **Tamaño muestral efectivo:** el número de pares no equivale necesariamente al número de observaciones independientes, debido a autocorrelación y variación de cadencia.
4. **Cobertura por energía:** un patrón con pocas mediciones válidas puede ser inestable; revisar siempre los mapas de cantidad de casos.
5. **Diferencia entre promedio de coeficientes y correlación agrupada:** se está promediando \(r\) entre cruces de forma descriptiva y no ponderada.

Por estos motivos, no se atribuye causalidad, ni se deduce una escala física característica solo por observar un cambio o una eventual meseta en un mapa.


## 10. Próximas etapas propuestas (aún no implementadas en los scripts)

- **Autocorrelación** de MAG y escalas temporales de persistencia por región.
- **PSD mediante Welch** para comparar potencia y pendientes espectrales antes y después de BS y MPB.
- **STFT y comparación wavelet MAG–SWEA** para estudiar cuándo aparecen fluctuaciones y si presentan relación entre instrumentos.
- **Correlación cruzada MAG–SWEA** en función de la energía y del retardo temporal.
- **Coherencia espectral** en intervalos con duración y resolución SWEA suficientes.
- **Control de contaminación entre fronteras** usando la separación BS–MPB en el catálogo clasificado.
- **Análisis de dispersión entre eventos** y comparación con una versión de señales sin tendencia.

Estos son planes de análisis, no resultados establecidos por el trabajo actual.


---

## 11. Reproducción de los resultados originales

Todos los comandos se ejecutan **desde la raíz del repositorio**. Las etapas con BigQuery pueden consumir cuota/costo y **no se ejecutan al abrir este cuaderno**.

**Instalación de dependencias:**

~~~bash
python3 -m pip install -r requirments.txt
~~~

**Control posterior de MAG ya tratado (solo archivos locales):**

~~~bash
python3 scripts/mag/01h_MAG_post_treatment_QC.py
~~~

**Revisar etiquetas de BS/MPB sin BigQuery:**

~~~bash
python3 scripts/correlations/04a_boundary_energy_correlation_maps.py \
  --events data/classified_crossings/ --catalog-only
~~~

**Estimación de lectura MAG–SWEA:**

~~~bash
python3 scripts/correlations/04a_boundary_energy_correlation_maps.py \
  --events data/classified_crossings/ --estimate-only
~~~

**Ejecución completa de correlaciones, de 2 a 60 minutos:**

~~~bash
python3 scripts/correlations/04a_boundary_energy_correlation_maps.py \
  --events data/classified_crossings/ --max-scan-gib 150
~~~

**Volver a generar gráficos de correlación sin consultar BigQuery:**

~~~bash
python3 scripts/correlations/04a_boundary_energy_correlation_maps.py \
  --events data/classified_crossings/ --plots-only
~~~

**CWT de MAG en una ventana total de 10 minutos alrededor de MPB (sin BigQuery):**

~~~bash
python3 scripts/wavelets/05a_MPB_MAG_wavelet_10min.py \
  --events data/classified_crossings/ --max-plots 12
~~~

La opción **--max-scan-gib** solo impone un límite interno al script; no modifica las cuotas de Google Cloud. **No usar --overwrite** si se desea reutilizar el caché diario disponible.

Para conocer el orden de construcción de los Parquet MAG, consultar los scripts **01–01h** enumerados en el README del repositorio.


## 12. Cierre del estado actual

La cadena de trabajo ya contiene:

- Caracterización de calidad temporal de MAG y SWEA.
- Estudio de discontinuidades y validación cuantitativa de reconstrucciones MAG.
- Tratamiento conservador de gaps y QC posterior.
- Cobertura espacial de las observaciones de MAVEN.
- Extracción de eventos BS y MPB clasificados.
- Sincronización temporal MAG–SWEA y análisis de Pearson dependiente de energía y duración de la ventana, con resúmenes por frontera y mapas de cantidad de casos.
- Primer análisis tiempo–frecuencia con CWT Morlet de MAG en ventanas de 10 minutos centradas en la MPB, con tratamiento de gaps y COI.

**Siguiente paso sugerido:** comparar PSD de Welch de MAG en intervalos representativos a ambos lados de BS/MPB y, luego, avanzar hacia STFT/CWT, manteniendo separadas las observaciones que correspondan a distintas regiones físicas.

**Nota de versión:** este notebook es una síntesis **a partir de los scripts y resultados ya archivados**. Las tablas de correlación y otras salidas no versionadas se completan automáticamente al ejecutarlo sobre la carpeta local que contiene esos resultados.
